In [0]:

from config.config import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType
import pycountry


In [0]:

# Build dim_country from live Silver tables.
# Sources: World Bank, UN inbound, UN accommodation, Eurostat.
wb = spark.read.format("delta").load(f"{silver_path}/world_bank_international_arrivals")
un_arr = spark.read.format("delta").load(f"{silver_path}/un_tourism_inbound_arrivals")
un_ac = spark.read.format("delta").load(f"{silver_path}/un_tourism_accommodation_hotels")
estat = spark.read.format("delta").load(f"{silver_path}/estat_tour_occ_arm")

wb_countries = wb.filter(F.col("country_iso3").isNotNull()).select(
    F.col("country_iso3").alias("iso3"), F.col("country_name").alias("name")).distinct()
wb_aggs = wb.filter(F.col("country_iso3").isNull()).select(
    F.lit("MAPPED_AGGREGATE").alias("mapping_status"),
    F.col("country_name").alias("name"), F.col("country_code").alias("code")).distinct()

un_reporters = un_arr.filter(F.col("reporter_country_iso3").isNotNull()).select(
    F.col("reporter_country_iso3").alias("iso3"), F.col("reporter_area").alias("name")).distinct()
un_partner_world = un_arr.filter(F.col("partner_geography_type") == "world_aggregate").select(
    F.lit(None).cast("string").alias("iso3"), F.lit("World").alias("name")).distinct()
un_ac_reporters = un_ac.filter(F.col("reporter_country_iso3").isNotNull()).select(
    F.col("reporter_country_iso3").alias("iso3"), F.col("reporter_area").alias("name")).distinct()
estat_countries = estat.filter(F.col("country_iso3").isNotNull()).select(
    F.col("country_iso3").alias("iso3"), F.col("geo_code").alias("name")).distinct()
estat_aggs = estat.filter(F.col("country_iso3").isNull()).select(
    F.lit("MAPPED_AGGREGATE").alias("mapping_status"),
    F.col("geo_code").alias("code"), F.lit("Eurostat aggregate").alias("name")).distinct()

countries = (wb_countries.unionByName(un_reporters).unionByName(un_ac_reporters)
             .unionByName(estat_countries)).distinct()


In [0]:

# Map ISO3 to M49, ISO2, region via pycountry; produce country_key = M49 for countries,
# negative surrogate for aggregates. Special territories get ISO3 codes (XKX, BES).
pyc_by_iso3 = {c.alpha_3: c for c in pycountry.countries}
pyc_by_m49 = {c.numeric: c for c in pycountry.countries}

def map_country(iso3):
    if not iso3:
        return None
    if iso3 in pyc_by_iso3:
        c = pyc_by_iso3[iso3]
        region = getattr(c, "region", None)
        return (int(c.numeric), c.alpha_2, iso3, c.name, region.name if region else "Unknown", True)
    return (None, None, iso3, iso3, "Unknown", True)

rows = []
for r in countries.collect():
    iso3 = r["iso3"]
    m = map_country(iso3)
    if m:
        rows.append((m[0], m[1], m[2], m[3], m[4], True, "MAPPED_COUNTRY"))
# Aggregates from World Bank and Eurostat
for r in wb_aggs.collect():
    rows.append((-1 - len(rows), None, r["code"], r["name"], "Aggregate", False, "MAPPED_AGGREGATE"))
for r in estat_aggs.collect():
    rows.append((-100 - len(rows), None, r["code"], r["name"], "Aggregate", False, "MAPPED_AGGREGATE"))
# Special territory WORLD (partner) and any UN special areas
for r in un_partner_world.collect():
    rows.append((-1000, None, None, "World", "Aggregate", False, "SPECIAL_GEOGRAPHY"))
# Dedup by ISO3 where possible
seen = {}
out = []
for r in rows:
    key = r[2] if r[2] else r[0]
    if key in seen:
        continue
    seen[key] = True
    out.append(r)

schema = StructType([
    StructField("country_key", IntegerType(), True),
    StructField("iso2", StringType(), True),
    StructField("iso3", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("region", StringType(), True),
    StructField("is_country", BooleanType(), True),
    StructField("mapping_status", StringType(), True),
])
dim_country = spark.createDataFrame(out, schema)
dim_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/dim_country")
display(dim_country.orderBy("country_name").limit(20))
print("dim_country rows:", dim_country.count())


In [0]:

# dim_date: monthly calendar 1990-01-01 .. 2026-12-01 + annual keys 1990-2027.
import datetime
rows = []
# monthly
d = datetime.date(1990, 1, 1)
end = datetime.date(2026, 12, 1)
i = 0
while d <= end:
    y, m = d.year, d.month
    rows.append((y * 100 + m, d, y, (m - 1) // 3 + 1, m,
                 datetime.date(d.year, d.month, 1).strftime("%B"),
                 ("Winter" if m in (12,1,2) else "Spring" if m in (3,4,5) else "Summer" if m in (6,7,8) else "Autumn"),
                 f"{y:04d}{m:02d}", 2012 <= y <= 2019))
    d = (datetime.date(y, m, 28) + datetime.timedelta(days=8)).replace(day=1)
# annual
for y in range(1990, 2028):
    rows.append((y, datetime.date(y, 1, 1), y, 1, None, None, None, None, 2012 <= y <= 2019))
from pyspark.sql.types import DateType
dim_date_schema = StructType([
    StructField("date_key", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("year", IntegerType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("month_name", StringType(), True),
    StructField("season", StringType(), True),
    StructField("year_month", StringType(), True),
    StructField("is_primary_period", BooleanType(), True),
])
dim_date = spark.createDataFrame([(r[0], r[1], r[2], r[3], r[4], r[5], r[6], r[7], r[8]) for r in rows], dim_date_schema)
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/dim_date")
display(dim_date.filter("is_primary_period = true").limit(10))
print("dim_date rows:", dim_date.count())
